# 01 · Criando o banco no SQL Server 2017

**Objetivo:** montar um banco que guarda embeddings **sem** o tipo `VECTOR` (que só existe a partir do SQL Server 2025).

**A ideia em uma frase:** um embedding é uma lista de números decimais (float32). Cada float32 ocupa 4 bytes. Então um vetor de 384 dimensões vira **exatamente 1.536 bytes**, e isso cabe perfeitamente num `VARBINARY`.

> Pré requisito: o arquivo `.env` da raiz do repositório existe e aponta para a sua instância de SQL Server (2017 ou mais novo).

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=UserWarning)   # aviso do pandas sobre conexão pyodbc
sys.path.append(str(pathlib.Path.cwd().parent))   # permite "from src import ..."
from src import config, banco

## 1. Conferindo a versão: estamos mesmo no 2017?

In [ ]:
conn_master = banco.conectar(autocommit=True)   # CREATE DATABASE exige autocommit
cur = conn_master.cursor()
print(cur.execute("SELECT @@VERSION").fetchval())

## 2. Prova de que o tipo VECTOR não existe aqui
Este erro é esperado. É exatamente o problema que a aula resolve.

In [ ]:
try:
    cur.execute("DECLARE @v VECTOR(3) = '[0.1, 0.2, 0.3]'; SELECT @v;")
except Exception as e:
    print("Como esperado, o 2017 não conhece VECTOR:\n", str(e)[:200])

## 3. Criando o banco

In [ ]:
banco.executar_script(conn_master, '''
IF DB_ID('RagPoliticos') IS NULL
    CREATE DATABASE RagPoliticos;
''')
conn_master.close()

conn = banco.conectar(config.SQL_DATABASE, autocommit=True)
print('Conectado em', config.SQL_DATABASE)

## 4. Schema `rag`
Separa os objetos da aula do resto do banco.

In [ ]:
banco.executar_script(conn, '''
IF SCHEMA_ID('rag') IS NULL
    EXEC('CREATE SCHEMA rag');
''')

## 5. O nosso "tipo vetor" caseiro: `rag.Embedding`

Um **alias type** não muda o armazenamento, mas deixa a intenção explícita no modelo de dados. Quem abrir a tabela vai ler `Vetor rag.Embedding` e entender na hora.

Por que `VARBINARY` e não `NVARCHAR` com JSON? Veremos a comparação de tamanho no notebook 02.

In [ ]:
banco.executar_script(conn, '''
-- Tipo de dado "apelido" (alias type): documenta a intenção da coluna.
-- Por baixo é VARBINARY(MAX): bytes crus de floats de 4 bytes.
IF TYPE_ID('rag.Embedding') IS NULL
    CREATE TYPE rag.Embedding FROM VARBINARY(MAX) NULL;
''')

## 6. Tabelas de negócio: políticos e frases

In [ ]:
banco.executar_script(conn, '''
IF OBJECT_ID('rag.Politico') IS NULL
CREATE TABLE rag.Politico (
    PoliticoId INT IDENTITY(1,1) CONSTRAINT PK_Politico PRIMARY KEY,
    Nome       NVARCHAR(150) NOT NULL CONSTRAINT UQ_Politico_Nome UNIQUE,
    Pais       NVARCHAR(60)  NOT NULL,
    Cargo      NVARCHAR(150) NULL
);
''')

In [ ]:
banco.executar_script(conn, '''
IF OBJECT_ID('rag.Frase') IS NULL
CREATE TABLE rag.Frase (
    FraseId        INT IDENTITY(1,1) CONSTRAINT PK_Frase PRIMARY KEY,
    PoliticoId     INT NOT NULL CONSTRAINT FK_Frase_Politico REFERENCES rag.Politico(PoliticoId),
    Texto          NVARCHAR(1000) NOT NULL,   -- texto em português (o que vamos vetorizar)
    TextoOriginal  NVARCHAR(1000) NULL,       -- idioma original da frase
    IdiomaOriginal CHAR(2)        NULL,
    Ano            SMALLINT       NULL,
    Contexto       NVARCHAR(300)  NULL,
    Fonte          NVARCHAR(400)  NULL
);
''')

## 7. Tabela de embeddings

Três decisões de DBA aqui:
1. **Tabela separada**: trocar de modelo não altera a tabela de negócio.
2. **Chave (FraseId, Modelo)**: vetores de modelos diferentes **não são comparáveis**; o modelo faz parte da identidade.
3. **CHECK de tamanho**: `DATALENGTH(Vetor) = Dimensoes * 4` impede gravar lixo. É a governança que o banco oferece e a aplicação sozinha não garante.

In [ ]:
banco.executar_script(conn, '''
-- Tabela separada para embeddings: permite ter mais de um modelo por frase
-- e re-vetorizar sem mexer na tabela de negócio.
IF OBJECT_ID('rag.FraseEmbedding') IS NULL
CREATE TABLE rag.FraseEmbedding (
    FraseId    INT           NOT NULL CONSTRAINT FK_FraseEmb_Frase REFERENCES rag.Frase(FraseId),
    Modelo     NVARCHAR(200) NOT NULL,
    Dimensoes  SMALLINT      NOT NULL,
    Vetor      rag.Embedding NOT NULL,
    CriadoEm   DATETIME2(0)  NOT NULL CONSTRAINT DF_FraseEmb_CriadoEm DEFAULT SYSUTCDATETIME(),
    CONSTRAINT PK_FraseEmbedding PRIMARY KEY (FraseId, Modelo),
    -- Governança: garante que o tamanho em bytes bate com as dimensões (float32 = 4 bytes)
    CONSTRAINT CK_FraseEmb_Tamanho CHECK (DATALENGTH(Vetor) = Dimensoes * 4)
);
''')

## 8. Bônus: tabela "explodida" para cálculo em T-SQL puro

In [ ]:
banco.executar_script(conn, '''
-- BÔNUS: o mesmo vetor "explodido" em linhas (uma linha por dimensão).
-- Serve para provar que dá para calcular o cosseno em T-SQL puro no 2017.
IF OBJECT_ID('rag.FraseEmbeddingItem') IS NULL
CREATE TABLE rag.FraseEmbeddingItem (
    FraseId INT           NOT NULL,
    Modelo  NVARCHAR(200) NOT NULL,
    Dim     SMALLINT      NOT NULL,
    Valor   REAL          NOT NULL,
    CONSTRAINT PK_FraseEmbeddingItem PRIMARY KEY (Modelo, Dim, FraseId)
);
''')

## 9. View e procedures

In [ ]:
banco.executar_script(conn, '''
CREATE OR ALTER VIEW rag.vw_Frases AS
SELECT f.FraseId, p.Nome AS Politico, p.Pais, p.Cargo,
       f.Texto, f.TextoOriginal, f.IdiomaOriginal, f.Ano, f.Contexto, f.Fonte
FROM rag.Frase f
JOIN rag.Politico p ON p.PoliticoId = f.PoliticoId;
''')

In [ ]:
banco.executar_script(conn, '''
-- Busca por palavra-chave: o jeito "clássico". Ignora acentos e maiúsculas (CI_AI).
CREATE OR ALTER PROCEDURE rag.usp_BuscaPalavraChave
    @Termo NVARCHAR(200)
AS
BEGIN
    SET NOCOUNT ON;
    SELECT FraseId, Politico, Texto, Ano
    FROM rag.vw_Frases
    WHERE Texto COLLATE Latin1_General_CI_AI LIKE N'%' + @Termo + N'%'
       OR TextoOriginal COLLATE Latin1_General_CI_AI LIKE N'%' + @Termo + N'%';
END;
''')

In [ ]:
banco.executar_script(conn, '''
-- BÔNUS: busca semântica em T-SQL puro (SQL Server 2016+), sem tipo VECTOR.
-- O vetor da pergunta chega como JSON; OPENJSON o transforma em linhas (Dim, Valor).
-- Como os vetores são normalizados, cosseno = soma dos produtos (produto escalar).
CREATE OR ALTER PROCEDURE rag.usp_BuscaSemanticaTSQL
    @VetorJson NVARCHAR(MAX),
    @Modelo    NVARCHAR(200),
    @K         INT = 5
AS
BEGIN
    SET NOCOUNT ON;
    WITH consulta AS (
        SELECT CAST([key] AS SMALLINT) AS Dim, CAST([value] AS REAL) AS Valor
        FROM OPENJSON(@VetorJson)
    )
    SELECT TOP (@K) v.FraseId, v.Politico, v.Texto,
           SUM(CAST(e.Valor AS FLOAT) * c.Valor) AS Similaridade
    FROM rag.FraseEmbeddingItem e
    JOIN consulta c ON c.Dim = e.Dim
    JOIN rag.vw_Frases v ON v.FraseId = e.FraseId
    WHERE e.Modelo = @Modelo
    GROUP BY v.FraseId, v.Politico, v.Texto
    ORDER BY Similaridade DESC;
END;
''')

## 10. Conferindo o que criamos

In [ ]:
import pandas as pd
pd.read_sql("""
SELECT s.name AS [schema], o.name AS objeto, o.type_desc AS tipo
FROM sys.objects o JOIN sys.schemas s ON s.schema_id = o.schema_id
WHERE s.name = 'rag' AND o.type IN ('U','V','P')
ORDER BY o.type_desc, o.name
""", conn)

In [ ]:
pd.read_sql("""
SELECT c.name AS coluna, t.name AS tipo, bt.name AS tipo_base, c.max_length
FROM sys.columns c
JOIN sys.types t  ON t.user_type_id = c.user_type_id
JOIN sys.types bt ON bt.user_type_id = t.system_type_id
WHERE c.object_id = OBJECT_ID('rag.FraseEmbedding')
""", conn)

Repare: a coluna `Vetor` aparece como tipo `Embedding`, com tipo base `varbinary` e `max_length = -1` (MAX).